In [ ]:
# PDF 로더

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("D:/shproject/ex0928/data/가상Tech_업무가이드.pdf")

documents = loader.load()

print(len(documents))
print(documents[0].page_content)


In [ ]:
# PDF 청킹

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

docs = text_splitter.split_documents(documents)

print(len(docs))
print(docs[0].page_content)

In [ ]:
# Embedding 모델 지정, FAISS로 벡터 DB 저장, 리트리버 까지
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
# LLM 객체 생성
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0
)

In [ ]:
# Prompt 지정

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
다음 문서를 참고하여 질문에 답변하세요.
문서에 없는 내용은 추측하거나 임의로 만들지 말고, 모른다고 답변하세요.

문서:
{context}

질문:
{question}

답변:
""")

In [ ]:
# 체인 생성

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt | llm | StrOutputParser()
)

In [ ]:
response = chain.invoke("제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요.")

print(response)